# 04 · Retrieval methods, embedding models, and reranking

**Deck section 4** · slides 33–50

Retrieval is staged: a fast first-stage method produces candidates, then you spend more
compute only where it improves ranking quality. This notebook builds every stage from
scratch — BM25 term by term, a navigable small-world graph you can watch lose recall, a
reranker you *fit* and then verify on a slice it never saw — and measures each one against
the same eval set.

Three of the results in here contradict the thing people expect. Those are the ones worth
your attention.

**By the end you can**

- write BM25 from memory and say what `k₁` and `b` do to a score
- explain why an analyzer setting can make identifiers unsearchable and nothing will tell you
- read an ANN recall curve, and know why a selective filter breaks a graph index
- diagnose a recall regression after an encoder swap in the order a strong candidate does it
- defend a fusion weight and a reranker with a bootstrap interval rather than a headline


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import re, time
import numpy as np
import pandas as pd
import raglab
from raglab import (viz, tables, catalog, chunking, corpus, embed, metrics,
                     pipeline, retrieve, store)
from raglab.embed import tokenize
viz.reset_figures("4."); tables.reset_tables("4.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)

# Sweeps run on a dev subsample so the notebook stays under a couple of minutes.
# The frozen slice is held back for the one measurement at the end that needs it.
dev = [q for q in bundle.questions if q.slice == "dev"]
sweep = dev[:90]
frozen = [q for q in bundle.questions if q.slice == "frozen" and q.question_type != "null"]
print(f"sweep set {len(sweep)} questions · frozen slice {len(frozen)} held back")


def qclass(q):
    '''Group questions by what they demand of a retriever.'''
    if re.search(r"ERR_[A-Z_]+", q.query):
        return "identifier"
    if "pause between" in q.query:
        return "pure lexical gap"
    if "headquartered in" in q.query or "-based" in q.query:
        return "descriptor (paraphrase)"
    if q.question_type == "null":
        return "null"
    return "named entity"


def by_class(rows, questions, key="evidence_recall"):
    for r, q in zip(rows, questions):
        r["qclass"] = qclass(q)
    return metrics.slice_report(rows, by="qclass", keys=(key,)).set_index("qclass")[key]

---

## 4.1 Staging is the whole idea

Cheap and wide first, expensive and narrow second. Every method below is a candidate for one
of these two slots, and the interesting question is always *which slot*, not *which method*.


In [ ]:
viz.stages([
    dict(stage="Stage 1 · O(corpus)", name="First-stage retrieval", tone="cool",
         body="Runs against every chunk. Must be cheap enough to do that: an inverted index "
              "lookup, or an approximate nearest-neighbour search.",
         knob="N, fusion weights, efSearch"),
    dict(stage="Stage 2 · O(N)", name="Reranking", tone="amber",
         body="Runs against N candidates only. Can afford to look at the query and the "
              "passage together, which is what makes it better and expensive.",
         knob="model class, candidate depth"),
], title="Two slots, and the cost model that decides what goes in each",
   kicker="Section 4 · staging",
   caption="A cross-encoder in stage one is a forward pass per chunk per query. That is not "
           "a tuning problem, it is an arithmetic one.",
   source="Deck slide 33")

---

## 4.2 Lexical retrieval, and BM25 from first principles

BM25 combines three ideas: rare terms count for more (IDF), the tenth occurrence of a term
adds far less than the second (saturation, `k₁`), and a long document should not win simply by
being long (length normalisation, `b`).

$$\text{score}(q,d) = \sum_i \text{IDF}(q_i)\cdot\frac{f(q_i,d)\,(k_1+1)}{f(q_i,d)+k_1\left(1-b+b\frac{|d|}{\text{avgdl}}\right)}$$

### What the code is about to do

Implement that formula directly, then check it ranks the same way SQLite's own FTS5
implementation does. If your from-scratch version disagrees with a production one, you have
learned something either way.


In [ ]:
viz.flow([("Documents", "the corpus"), ("Analyze & tokenize", "the step that decides what is "
          "searchable at all"), ("Inverted index", "term → postings"),
          ("BM25 scoring", "IDF × saturation × length norm"), ("Top-k documents", "ranked")],
         title="Lexical retrieval: inverted index and BM25",
         kicker="Section 4 · lexical",
         caption="Deterministic, explainable, inexpensive, and easy to filter by metadata. "
                 "Its one limitation is the lexical gap.",
         source="Deck slide 35")

In [ ]:
def bm25(query, docs, k1=1.5, b=0.75):
    '''BM25, written out so every term is visible.'''
    import math
    tokd = [tokenize(d) for d in docs]
    N = len(docs)
    avgdl = sum(len(t) for t in tokd) / max(1, N)

    df = {}
    for toks in tokd:
        for t in set(toks):
            df[t] = df.get(t, 0) + 1

    scores = []
    for toks in tokd:
        tf = {}
        for t in toks:
            tf[t] = tf.get(t, 0) + 1
        dl = len(toks)
        s = 0.0
        for t in tokenize(query):
            f = tf.get(t, 0)
            if not f:
                continue
            idf = math.log(1 + (N - df.get(t, 0) + 0.5) / (df.get(t, 0) + 0.5))
            saturation = (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))
            s += idf * saturation
        scores.append(s)
    return scores


probe = "Which organization acquired Tessera Analytics?"
pool = [c for c in pipe.chunks[:400]]
mine = bm25(probe, [c.text for c in pool])
top_mine = sorted(zip(mine, pool), key=lambda t: -t[0])[:5]

print("FROM SCRATCH")
for s, c in top_mine:
    print(f"  {s:7.3f}  {c.doc_id:<10} {c.text[:62]}")

fts = index.lexical(probe, n=5)
print("\nSQLITE FTS5 bm25()")
for h in fts:
    print(f"  {h.score:7.3f}  {h.doc_id:<10} {h.text[:62]}")

overlap = len({c.chunk_id for _, c in top_mine} & {h.chunk_id for h in fts})
print(f"\ntop-5 overlap on a shared 400-chunk pool: {overlap}/5")
print("(FTS5 scores the full index and weights the title field, so the two are not required")
print(" to agree exactly — what should agree is the ordering logic, not the constants.)")

In [ ]:
# What k1 and b actually do, on one query.
target = top_mine[0][1]
rows = []
for k1 in (0.5, 1.2, 2.0, 5.0):
    for bb in (0.0, 0.75, 1.0):
        s = bm25(probe, [target.text], k1=k1, b=bb)[0]
        rows.append({"k₁": k1, "b": bb, "score": round(s, 3)})
grid = pd.DataFrame(rows).pivot(index="k₁", columns="b", values="score")
tables.show(grid.reset_index(), title="One chunk, one query, nine parameter settings",
            kicker="BM25 knobs",
            caption="k₁ controls term-frequency saturation: raise it and repeated terms keep "
                    "adding. b controls length normalisation: b=0 ignores document length "
                    "entirely, b=1 fully normalises it.",
            emphasize="k₁")

tables.callout(
    "<b>Chunking changes avgdl.</b> Re-chunking silently re-tunes every BM25 score in your "
    "index, because the length-normalisation term is relative to the average document length "
    "of the corpus you just rebuilt. Re-measure lexical recall after every chunking change — "
    "the code did not change, and the ranking did.", kind="warn")

---

## 4.3 The analyzer trap

Before any of that arithmetic runs, a tokenizer decides what a term *is*. That setting is
usually left at its default, and on a corpus with identifiers the default is wrong.


In [ ]:
import sqlite3

def probe_tokenizer(tokenchars):
    db = sqlite3.connect(":memory:")
    tok = "unicode61 remove_diacritics 2" + (f" tokenchars '{tokenchars}'" if tokenchars else "")
    db.execute(f'CREATE VIRTUAL TABLE t USING fts5(x, tokenize = "{tok}")')
    for c in pipe.chunks:
        db.execute("INSERT INTO t VALUES (?)", (c.text,))
    hits = db.execute('SELECT count(*) FROM t WHERE t MATCH ?', ('"ERR_CONN_RESET"',)).fetchone()[0]
    return hits

default_hits = probe_tokenizer(None)
fixed_hits = probe_tokenizer("_-")
actual = sum(1 for c in pipe.chunks if "ERR_CONN_RESET" in c.text)

tables.show(pd.DataFrame([
    ["unicode61 (the default)", default_hits,
     "ERR_CONN_RESET is split into err / conn / reset, and those three tokens appear in every "
     "incident report in the corpus"],
    ["unicode61 tokenchars '_-'", fixed_hits,
     "the identifier survives as one token and matches exactly"],
    ["ground truth (substring scan)", actual, "the number of chunks that literally contain it"],
], columns=["Analyzer", "Chunks matched for \"ERR_CONN_RESET\"", "What happened"]),
    title="One tokenizer setting decides whether identifiers are searchable",
    kicker="The analyzer trap",
    caption="Nothing errors. The query runs, returns results, and they are the wrong results. "
            "This is the single cheapest silent recall bug in enterprise search.",
    emphasize="Chunks matched for \"ERR_CONN_RESET\"")

In [ ]:
ident = [q for q in bundle.questions if re.search(r"ERR_[A-Z_]+", q.query)]
ident_rows = pipeline.evaluate(pipe.variant("lex", fusion="lexical", rerank="none"),
                               ident, pipe.chunks, personas=bundle.personas)
print(f"identifier questions ({len(ident)}), lexical leg only")
print(f"  evidence recall with the corrected analyzer: "
      f"{metrics.summarize(ident_rows)['evidence_recall']:.3f}")
print("\nThis toolkit ships with tokenchars '_-' set in store.py. The comment above that line")
print("is longer than the line, and that ratio is correct.")

---

## 4.4 Semantic retrieval: the encoder, the index, and one identity

Encode documents offline in batch; encode the query online per request. That asymmetry is
what makes dense retrieval affordable at all — you pay the document cost once.


In [ ]:
viz.hld([
    dict(name="Offline · once per chunk version", tone="index", nodes=[
        ("Document chunk", "text"), ("Document encoder f_d", "batched"),
        ("d ∈ ℝᵈ", "L2-normalised"), ("ANN index", "HNSW / IVF, built once")]),
    dict(name="Online · once per request", tone="query", nodes=[
        ("Query", "text"), ("Query encoder f_q", "one forward pass"),
        ("q ∈ ℝᵈ", "L2-normalised"), ("Nearest neighbours", "top-N by cosine")]),
], title="Bi-encoder search: two towers, one shared space",
   kicker="Section 4 · semantic",
   caption="The two encoders may share weights or use different query/document instructions. "
           "Mismatched instructions are a silent recall bug — §4.8 reproduces it.",
   source="Deck slides 37–38")

In [ ]:
# cos(q,d) = q·d / (‖q‖‖d‖). After L2 normalisation, that is just q·d.
q_raw = pipe.embedder._embed(["Which organization acquired Tessera Analytics?"], "")
d_raw = pipe.embedder._embed([pipe.chunks[0].text, pipe.chunks[7].text], "")

def cosine_longhand(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for i in range(2):
    long_form = cosine_longhand(q_raw[0], d_raw[i])
    dot_form = float(np.dot(q_raw[0], d_raw[i]))
    print(f"  chunk {i}   cos = {long_form:+.6f}   dot = {dot_form:+.6f}   "
          f"identical: {abs(long_form - dot_form) < 1e-6}")

print(f"\n  ‖q‖ = {np.linalg.norm(q_raw[0]):.6f}   ‖d‖ = {np.linalg.norm(d_raw[0]):.6f}")
print("\nThat is the whole of the identity, and it has a real engineering consequence: an")
print("index configured for inner product returns the same ranking as one configured for")
print("cosine ONLY if the vectors are normalised. Normalise on write, or configure the index")
print("for cosine, and know which one you did.")

In [ ]:
# Scores are relative, not calibrated. A threshold does not transfer.
qs = [q.query for q in sweep[:40]]
tops = []
for text in qs:
    hits = retrieve.DenseRetriever(index, pipe.embedder).search(
        text, 10, retrieve.RetrievalConfig(n_candidates=10))
    if hits:
        tops.append(hits[0].score)

print(f"top-1 cosine across 40 queries on ONE corpus:")
print(f"  min {min(tops):.3f}   median {sorted(tops)[len(tops)//2]:.3f}   max {max(tops):.3f}")
print(f"  spread {max(tops)-min(tops):.3f}")
print("\nA fixed similarity cut-off is not portable across corpora, models, or even reindexes.")
print("If you need a threshold, calibrate it per deployment against labelled examples —")
print("and notebook 06 shows what happens when you try to use one for abstention.")

---

## 4.5 Dimension is a generalisation knob, not only a cost knob

The deck's embedding tree tells you to *measure the recall you lose at 512 and 256 dims
before paying for 3072*. That framing treats dimension as a cost decision with a quality
penalty. It is also the opposite: aggressive reduction *smooths* the space and starts
bridging vocabulary, which is the entire mechanism by which dense retrieval beats keyword
matching.

Sweep it, and slice by what each question demands.


In [ ]:
dims = [24, 48, 96, 192, 384]
overall, per_class = [], {}
for d in dims:
    em = embed.LsaEmbedder(dim=d).fit([x.title + "\n" + x.body for x in bundle.documents])
    idx = store.InMemoryIndex()
    idx.upsert(pipe.chunks, em.encode_documents([c.text for c in pipe.chunks]), "v1",
               em.info.tag)
    p = pipeline.RagPipeline(idx, em, retrieve.RetrievalConfig(
        n_candidates=100, k=8, fusion="dense", rerank="none"), name=f"d={d}")
    rs = pipeline.evaluate(p, sweep, pipe.chunks, personas=bundle.personas)
    overall.append(metrics.summarize(rs)["evidence_recall"])
    sl = by_class(rs, sweep)
    for cls, v in sl.items():
        per_class.setdefault(cls, []).append(v)

series = {"all questions": overall}
series.update({k: v for k, v in per_class.items()
               if k in ("descriptor (paraphrase)", "identifier", "named entity")})
viz.lines(dims, series,
          title="Embedding dimension trades exactness against generalisation",
          kicker="Measured · dense leg only",
          xlabel="latent dimensions", ylabel="evidence recall @8",
          caption="Paraphrase questions peak low, where the space is smoothed hard. "
                  "Exact-match questions peak high, where it reproduces the term space. "
                  "One number cannot be right for both — which is what routing is for.")

In [ ]:
frame = pd.DataFrame({"dimensions": dims, "all": [round(v, 3) for v in overall]})
for k, v in per_class.items():
    frame[k] = [round(x, 3) if x == x else None for x in v]
tables.show(frame, title="The same sweep, read as a decision",
            kicker="Dimension sweep",
            caption="Pick the dimension your dominant query class needs, then check what it "
                    "cost the others. Reporting only the 'all' column hides the trade "
                    "entirely — which is the deck's point about averages, in miniature.",
            emphasize="all")

---

## 4.6 Approximate search: real graph, real recall loss

`store.ann_vector()` is a navigable small-world search — a k-NN graph, multiple entry points,
greedy best-first expansion with a bounded visit budget. Nothing here is simulated, which
means the recall curve is a measurement rather than an assertion.


In [ ]:
probe_q = "Who is the chief executive of the company that acquired Tessera Analytics?"
qv = pipe.embedder.encode_queries([probe_q])[0]
exact = index.exact_vector(qv, n=20)
exact_ids = {h.chunk_id for h in exact}

efs = [8, 16, 32, 64, 128, 256, 512]
recall, visits = [], []
for ef in efs:
    ann = index.ann_vector(qv, n=20, ef_search=ef)
    recall.append(len({h.chunk_id for h in ann} & exact_ids) / 20)
    visits.append(index.last_ann_visits)

viz.lines(efs, {"recall@20 vs exact search": recall},
          title="ANN recall is a tunable, not a property",
          kicker="Measured · navigable small-world graph",
          xlabel="efSearch (visit budget)", ylabel="recall against flat search",
          caption=f"Flat search over {len(pipe.chunks):,} chunks is the ground truth. "
                  "At low efSearch the index misses chunks whose embeddings were perfectly "
                  "correct — and no downstream metric will tell you that is why.")

tables.show(pd.DataFrame({
    "efSearch": efs, "nodes visited": visits,
    "recall@20": [round(r, 3) for r in recall],
    "share of corpus scanned": [f"{v/len(pipe.chunks):.1%}" for v in visits],
}), title="What you buy with each additional visit",
   kicker="ANN cost/recall",
   caption="Flat search visits 100% of the corpus for recall 1.0. The whole value of an ANN "
           "index is the shape of this curve — and the whole risk is shipping the left end "
           "of it by accident.", emphasize="recall@20")

In [ ]:
# Filtered search is where graph indexes degrade -- the deck's warning, measured.
analyst = bundle.personas["analyst"]
counsel = bundle.personas["counsel"]

rows = []
for label, groups, mode in [
    ("no filter", None, "pre"),
    ("broad filter (counsel: sees everything)", counsel, "pre"),
    ("selective filter (analyst: public only)", analyst, "pre"),
    ("very selective filter (one source)", None, "pre"),
]:
    filters = {"source": "incident"} if label.startswith("very") else None
    exact_f = index.exact_vector(qv, n=20, acl_groups=groups, filters=filters)
    ids_f = {h.chunk_id for h in exact_f}
    ann_f = index.ann_vector(qv, n=20, ef_search=64, acl_groups=groups, filters=filters,
                             filter_mode="pre")
    got = {h.chunk_id for h in ann_f}
    rows.append([label, len(ids_f), len(got),
                 round(len(got & ids_f) / max(1, len(ids_f)), 3), index.last_ann_visits])

tables.show(pd.DataFrame(rows, columns=[
    "Filter", "Flat search finds", "ANN returns", "ANN recall vs flat", "Nodes visited"]),
    title="A selective pre-filter strands the graph traversal",
    kicker="Filtered ANN",
    caption="The traversal can only walk through nodes it is allowed to see. When those are "
            "sparse, it dead-ends in a region far from the answer — at the same efSearch. "
            "Test recall with your real filters, not without them.",
    emphasize="ANN recall vs flat",
    highlight_rows=lambda r: r["ANN recall vs flat"] < 0.9)

In [ ]:
catalog.ANN_INDEX.show()

---

## 4.7 Choosing an embedding model, and surviving the swap

The tree first. Then the failure that actually happens when you follow it.


In [ ]:
env = {
    "corpus_can_leave_network": True,
    "domain_vocabulary_far": False,
    "multilingual": False,
    "large_index": len(pipe.chunks) > 10_000_000,
}
catalog.EMBEDDING_TREE.explain(env)

In [ ]:
catalog.EMBEDDING_TREE.show_table()

### Interview Q3, executed

> *"You upgraded the embedding model and Evidence Recall@10 fell from 0.86 to 0.71. Walk me
> through the diagnosis."*

A 15-point drop is almost never "the new model is worse". A strong candidate checks
operational causes before model-quality causes, in a fixed order, and bisects rather than
guesses. Every step below is reproducible on this index.


In [ ]:
baseline = pipeline.evaluate(pipe.variant("dense", fusion="dense", rerank="none"),
                             sweep, pipe.chunks, personas=bundle.personas)
base_recall = metrics.summarize(baseline)["evidence_recall"]

diagnosis = []

# 1 · Mixed-version index.
mixed = index.mixed_version_check("v1")
diagnosis.append(["1 · Mixed-version index",
                  "Are some vectors still from the old model?",
                  f"embedder tags present: {list(mixed['tags'])} → {'clean' if mixed['ok'] else 'MIXED'}",
                  "One SQL query against the model-version tag. Do this first because it is "
                  "the cheapest and the most common."])

# 2 · Prefix asymmetry -- reproduced.
em_pref = embed.LsaEmbedder(dim=96, doc_prefix="passage: ", query_prefix="query: ").fit(
    [d.title + "\n" + d.body for d in bundle.documents])
idx_pref = store.InMemoryIndex()
idx_pref.upsert(pipe.chunks, em_pref.encode_documents([c.text for c in pipe.chunks]), "v1",
                em_pref.info.tag)
correct = pipeline.evaluate(pipeline.RagPipeline(idx_pref, em_pref, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none")), sweep, pipe.chunks,
    personas=bundle.personas)
em_broken = embed.LsaEmbedder(dim=96, doc_prefix="passage: ", query_prefix="")
em_broken.vocab, em_broken.idf = em_pref.vocab, em_pref.idf
em_broken.V, em_broken.S, em_broken.term_space = em_pref.V, em_pref.S, em_pref.term_space
em_broken.dim = em_pref.dim
broken = pipeline.evaluate(pipeline.RagPipeline(idx_pref, em_broken, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none")), sweep, pipe.chunks,
    personas=bundle.personas)
diagnosis.append(["2 · Prefix asymmetry",
                  "Does the model need query/passage prefixes, applied on the right side?",
                  f"correct {metrics.summarize(correct)['evidence_recall']:.3f} → "
                  f"query prefix dropped {metrics.summarize(broken)['evidence_recall']:.3f}",
                  "Documents carry the passage prefix, queries lose the query prefix. "
                  "Nothing errors. Recall just falls."])

# 3 · Normalisation and metric.
unnorm = pipe.embedder._embed([sweep[0].query], "")
diagnosis.append(["3 · Normalisation and metric",
                  "Are the new vectors L2-normalised, and is the index still on cosine?",
                  f"‖v‖ = {np.linalg.norm(unnorm[0]):.4f} → normalised",
                  "Inner-product and cosine agree only on normalised vectors. Mixing them "
                  "reorders the whole ranking silently."])

# 4 · Dimension truncation.
em_trunc = embed.LsaEmbedder(dim=24).fit([d.title + "\n" + d.body for d in bundle.documents])
idx_t = store.InMemoryIndex()
idx_t.upsert(pipe.chunks, em_trunc.encode_documents([c.text for c in pipe.chunks]), "v1",
             em_trunc.info.tag)
trunc = pipeline.evaluate(pipeline.RagPipeline(idx_t, em_trunc, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none")), sweep, pipe.chunks,
    personas=bundle.personas)
diagnosis.append(["4 · Dimension truncation",
                  "Was the model shortened to fit the existing schema?",
                  f"96 dims {base_recall:.3f} → 24 dims "
                  f"{metrics.summarize(trunc)['evidence_recall']:.3f}",
                  "A schema that will not change is a very common reason a model gets "
                  "truncated without anyone measuring the cost."])

# 5 · Context-length truncation.
long_chunks = [c for c in pipe.chunks if chunking.approx_tokens(c.text) > 200]
diagnosis.append(["5 · Context-length truncation",
                  "Does the new encoder have a shorter input limit?",
                  f"{len(long_chunks)} of {len(pipe.chunks)} chunks exceed 200 tokens "
                  f"({len(long_chunks)/len(pipe.chunks):.0%})",
                  "An encoder with a 256-token window silently cuts the tail off every one "
                  "of those, and the tail is often where the answer is."])

# 6 · ANN parameters -- compare against flat search to isolate index loss.
ann_p = pipeline.RagPipeline(index, pipe.embedder, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none", ann=True, ef_search=32))
ann_rows = pipeline.evaluate(ann_p, sweep, pipe.chunks, personas=bundle.personas)
diagnosis.append(["6 · ANN parameters",
                  "Was the graph rebuilt with the same parameters? Compare against flat.",
                  f"flat {base_recall:.3f} → ANN(ef=32) "
                  f"{metrics.summarize(ann_rows)['evidence_recall']:.3f}",
                  "Flat search is the ground truth that separates index loss from embedding "
                  "loss. A candidate who does not know this cannot bisect."])

diagnosis.append(["7 · The model really is worse here",
                  "Only now. Slice the misses by question type to show where.",
                  "see §4.5 — the dimension sweep is this step done properly",
                  "Starting at step 7 is the red flag the panel is listening for."])

tables.show(pd.DataFrame(diagnosis, columns=[
    "Step", "The question", "Measured on this index", "Why it comes in this position"]),
    title="Diagnosing a recall regression after an encoder swap",
    kicker="Interview Q3 · executed",
    caption="Six operational causes before one model-quality cause. Every step above is a "
            "measurement you can run in under a minute on your own index.",
    source="Deck slide 92", emphasize="Step")

---

## 4.8 Grep, and two shipped products that disagree

`rg -n -i "retry|backoff" src tests`. Literal or regex matching over files, with paths and
line numbers. Precise and inspectable; it does not bridge paraphrases or rank evidence.
For an agent working on a repository, that trade is usually the right one.


In [ ]:
grep = retrieve.GrepRetriever(pipe.chunks)
for pattern in (r"ERR_[A-Z_]+", r"backoff interval", r"retry delay"):
    hits = grep.search(pattern, n=3)
    print(f"\n  rg '{pattern}'  →  {len(hits)} chunks")
    for h in hits[:2]:
        print(f"      {h.doc_id:<10} {h.text[:66]}")

print("\n\nPerfect precision on what it matched. Zero recall for anything phrased differently:")
print("'pause between reconnection attempts' returns nothing, and a dense leg finds it.")

In [ ]:
tables.show(pd.DataFrame([
    ["Wins on", "“Where is rate limiting handled?” — conceptual queries with no shared "
     "vocabulary", "Exactness and freshness — it reads the file as it is on disk right now"],
    ["Pays for", "Index freshness on every branch switch; a remote embedding step that "
     "touches customer source; stale results after a refactor",
     "Multiple model turns per question: higher latency, higher token cost, and a hard "
     "dependency on the agent's search strategy"],
    ["Retrieval state", "A precomputed index synced to the working tree",
     "The transcript. Nothing is precomputed."],
    ["Failure the client feels", "A stale answer", "A slow answer"],
], columns=["Dimension", "A · pre-indexed semantic retrieval", "B · agentic grep, no index"]),
    title="Code assistants: the same task, two production architectures",
    kicker="Case study · architecture divergence",
    caption="Neither is wrong. Indexed retrieval buys latency and conceptual recall at the "
            "price of a synchronisation problem; agentic grep buys correctness and zero index "
            "operations at the price of tokens and turns. Choose by asking which failure the "
            "client cannot tolerate.",
    source="Deck slide 44", emphasize="Dimension")

---

## 4.9 Hybrid fusion — and a result that contradicts the default

Two ways to merge two ranked lists. RRF ignores scores and uses ranks, which is robust and
needs no tuning. Weighted score fusion keeps magnitude, which is more expressive and needs a
labelled set to tune.

The deck's advice is **default to RRF**, and move to weighted fusion only with a labelled set
big enough to tune α. We have one. So let us find out whether the default survives.


In [ ]:
legs = {}
for name, kw in (("BM25 only", dict(fusion="lexical")), ("dense only", dict(fusion="dense")),
                 ("RRF (equal weight)", dict(fusion="rrf"))):
    legs[name] = pipeline.evaluate(pipe.variant(name, rerank="none", **kw), sweep,
                                   pipe.chunks, personas=bundle.personas)

alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
alpha_er, alpha_fcr = [], []
for a in alphas:
    rs = pipeline.evaluate(pipe.variant(f"a={a}", fusion="weighted", alpha=a, rerank="none"),
                           sweep, pipe.chunks, personas=bundle.personas)
    s = metrics.summarize(rs)
    alpha_er.append(s["evidence_recall"]); alpha_fcr.append(s["full_chain_recall"])
    legs[f"weighted α={a}"] = rs

best_a = alphas[int(np.argmax(alpha_er))]
viz.lines(alphas, {"evidence recall @8": alpha_er, "full-chain recall": alpha_fcr},
          title="Weighted fusion: sweeping the dense leg's share",
          kicker="Measured · α sweep", xlabel="α (dense share; 0 = pure BM25, 1 = pure dense)",
          ylabel="recall", vline=best_a, vline_label=f"best α={best_a}",
          caption="α=0 and α=1 are the single legs. Everything between them is the hybrid, "
                  "and the shape of this curve is the argument for tuning rather than "
                  "assuming.")

In [ ]:
summary = pipeline.compare_runs(
    {k: v for k, v in legs.items()
     if k in ("BM25 only", "dense only", "RRF (equal weight)", f"weighted α={best_a}")},
    keys=("evidence_recall", "full_chain_recall", "context_precision"))
tables.show(summary, title="Four ways to produce a candidate set",
            kicker="Fusion, measured", emphasize="run",
            caption="Read the RRF row against the BM25 row before you accept the default.")

rrf_er = metrics.summarize(legs["RRF (equal weight)"])["evidence_recall"]
lex_er = metrics.summarize(legs["BM25 only"])["evidence_recall"]
wt_er = metrics.summarize(legs[f"weighted α={best_a}"])["evidence_recall"]
print(f"BM25 alone            {lex_er:.3f}")
print(f"RRF (equal weight)    {rrf_er:.3f}   ({rrf_er - lex_er:+.3f} vs BM25 alone)")
print(f"weighted α={best_a}          {wt_er:.3f}   ({wt_er - lex_er:+.3f} vs BM25 alone)")

In [ ]:
tables.callout(
    "<b>On this corpus, fusion does not separate from its better single leg.</b> "
    "Both fusion rules beat BM25 alone decisively \u2014 <code>bm25 &rarr; rrf</code> is "
    "+0.0624 evidence recall, ci (+0.0407, +0.0857). Neither beats the <i>dense leg on its "
    "own</i>: <code>dense &rarr; rrf</code> is +0.0008 with an interval of (\u22120.0101, "
    "+0.0109), and on nDCG the unfused dense leg wins outright by 0.075."
    "<br><br>The mechanism is <b>complementarity, not comparable strength</b>. Fusion turns two "
    "signals into a better one only when the legs fail on <i>different</i> queries; two "
    "retrievers that fail together carry one signal between them. Cormack\u2019s RRF paper "
    "fuses TREC runs \u2014 mature systems that are good in different ways \u2014 and that "
    "property is doing more work in the paper than the equal weighting is."
    "<br><br>Here BM25 is the weak leg, not the dense one: these questions are paraphrase and "
    "inference over incident prose, where term overlap has almost nothing to score. BM25\u2019s "
    "genuine win is the exact-identifier slice, which is real and small."
    "<br><br>The measurement that would have settled this in advance is the <b>per-query overlap "
    "of the two legs\u2019 failures</b>, and nobody ran it before choosing. Disjoint failures: "
    "fusion is worth a lot. Nested failures: worth nothing."
    "<br><br>What you must not do is quote \u03b1 from this notebook. It is fitted to this "
    "corpus and this encoder, it is not even the argmax (\u03b1 = 0.5 measures better), and an "
    "\u03b1 tuned on 200 examples will not survive a corpus refresh.",
    kind="warn", title="Read this before you copy the number")

tables.callout(
    "This cell said the opposite until 2026-09-01. It read <i>\u201cequal-weight RRF does not "
    "beat BM25 alone, and weighted fusion does\u201d</i>, with a mechanism about fusing a "
    "strong leg with a weak one. It does not reproduce: RRF beats BM25, and the "
    "fifty-year-old dense leg is the stronger of the two."
    "<br><br>It was wrong for months and quoted in about twenty places. What let it survive is "
    "structural and worth more than the finding: the eval gate compares one configuration "
    "against its own history and never against alternatives, so a claim about which "
    "<i>configuration</i> wins sat outside everything CI could check. Reproduce the corrected "
    "table yourself with <code>python scripts/run_eval.py --compare</code>.",
    kind="note", title="Correction \u2014 ADR-0015")

In [ ]:
# Where each leg actually wins -- the reason a hybrid exists at all.
cls_rows = {}
for name in ("BM25 only", "dense only", f"weighted α={best_a}"):
    cls_rows[name] = by_class(legs[name], sweep)
cls = pd.DataFrame(cls_rows)
cls["n"] = metrics.slice_report(legs["BM25 only"], by="qclass",
                                keys=("evidence_recall",)).set_index("qclass")["n"]
tables.show(cls.reset_index().rename(columns={"index": "query class"}),
            title="Evidence recall by what the question demands",
            kicker="Why hybrid exists",
            caption="The legs fail in different places. That is the entire argument for "
                    "running both — and the reason a single global α is a compromise rather "
                    "than an optimum. Routing α by query class is the next move.",
            emphasize="query class")

---

## 4.10 Reranking: a model you fit, and then have to verify

A cross-encoder scores the (query, passage) pair with full attention over both. It is better
than a bi-encoder for a structural reason — a bi-encoder must compress the passage into one
vector *before it has seen the query* — and expensive for the same reason: nothing can be
precomputed, and cost is linear in N.

This toolkit's reranker is a logistic regression over eight pair features. Far weaker than a
trained transformer, architecturally the same animal, and — usefully — small enough that you
can fit it in front of yourself and watch it overfit.


In [ ]:
viz.flow([("Query + candidate passage", "concatenated, as a pair"),
          ("[CLS] q [SEP] passage [SEP]", "one sequence"),
          ("Early interaction", "query and passage tokens attend to each other"),
          ("Relevance score", "one forward pass per pair — nothing to index")],
         title="Early-interaction reranking, and its cost model",
         kicker="Section 4 · rerankers",
         caption="Reranking 100 candidates costs 100 forward passes, per query, every query. "
                 "Batch, do not loop: the difference is often 90 ms versus 900 ms.",
         source="Deck slide 45")

In [ ]:
# Fit the reranker on the DEV slice only, then verify on the frozen slice.
first_stage = pipe.variant("first", fusion="weighted", alpha=raglab.TUNED["alpha"],
                           rerank="none")
train_pairs = []
for q in dev:
    if q.question_type == "null":
        continue
    hits = first_stage.retriever.search(q.query, first_stage.cfg)[:50]
    gold = metrics.gold_chunk_ids(q, pipe.chunks)
    if gold:
        train_pairs.append((q.query, hits, gold))

t0 = time.perf_counter()
ce = retrieve.ProxyCrossEncoder(pipe.embedder,
                                weights={f: 0.0 for f in retrieve.PAIR_FEATURES})
ce.fit(train_pairs)
print(f"fitted on {len(train_pairs)} dev questions in {time.perf_counter()-t0:.1f}s\n")

w = ce.learned_weights
viz.bars(list(retrieve.PAIR_FEATURES),
         {"learned weight": [w[f] for f in retrieve.PAIR_FEATURES]},
         title="What the reranker learned to care about",
         kicker="Fitted coefficients", ylabel="logistic regression weight",
         caption="Negative weights are as informative as positive ones: a strong negative on "
                 "`length` means the model learned to distrust long chunks, which is exactly "
                 "what BM25's b parameter does by hand.")

In [ ]:
# Verify on the frozen slice -- the one thing tuning never saw.
def run_with(reranker, questions, name):
    v = pipe.variant(name, fusion="weighted", alpha=raglab.TUNED["alpha"], rerank="none")
    v.reranker = reranker
    return pipeline.evaluate(v, questions, pipe.chunks, personas=bundle.personas)

comparison = {}
for label, rr in (("no reranker", retrieve.NoReranker()),
                  ("late interaction (MaxSim)", retrieve.LateInteractionReranker(pipe.embedder)),
                  ("learned cross-encoder", ce)):
    comparison[f"{label} · dev"] = run_with(rr, sweep, label)
    comparison[f"{label} · frozen"] = run_with(rr, frozen, label)

tables.show(pipeline.compare_runs(comparison, keys=("evidence_recall", "full_chain_recall",
                                                    "context_precision")),
            title="Reranking, on the slice it was fitted on and the slice it never saw",
            kicker="Fit and verify",
            caption="A gain that appears on dev and vanishes on frozen is overfitting. A gain "
                    "that holds on both is a result. This is the discipline the build rubric "
                    "weights at 25%.",
            emphasize="run",
            highlight_rows=lambda r: "frozen" in r["run"] and "cross" in r["run"])

In [ ]:
for metric in ("evidence_recall", "full_chain_recall"):
    b = metrics.paired_bootstrap(comparison["no reranker · dev"],
                                 comparison["learned cross-encoder · dev"], metric)
    print(f"{metric:<20} delta {b['delta']:+.4f}   95% CI [{b['ci'][0]:+.4f}, "
          f"{b['ci'][1]:+.4f}]   p(better) {b['p_better']:.3f}   → {b['verdict']}")
print()
print("Read those two lines together. The same change is a real improvement on evidence")
print("recall and indistinguishable from zero on full-chain recall, on the same questions.")
print("Neither line is the 'right' one — they answer different questions, and a release gate")
print("that only watches one of them will ship or block for the wrong reason.")

In [ ]:
# The ceiling, stated as an inequality and then checked.
ceilings, delivered = [], []
for q in sweep:
    if q.question_type == "null":
        continue
    gm, _ = metrics.resolve_gold(q, pipe.chunks)
    if not gm:
        continue
    cands = first_stage.retriever.search(q.query, first_stage.cfg)
    ceilings.append(metrics.evidence_recall_at_k([h.chunk_id for h in cands], gm))
    reranked = ce.rerank(q.query, cands, depth=50)
    delivered.append(metrics.evidence_recall_at_k([h.chunk_id for h in reranked[:8]], gm))

violations = sum(1 for c, d in zip(ceilings, delivered) if d > c + 1e-9)
print(f"questions checked                  {len(ceilings)}")
print(f"mean Recall@N (the ceiling)        {sum(ceilings)/len(ceilings):.3f}")
print(f"mean Recall@k after reranking      {sum(delivered)/len(delivered):.3f}")
print(f"cases where reranking exceeded the ceiling   {violations}")
print("\nZero, and it must be zero: no stage-2 model can rank a document it never received.")
print("If Recall@N is 0.78, reranking cannot take end-to-end evidence recall above 0.78 —")
print("it can only reorder what survived stage one.")

In [ ]:
catalog.RERANKER_MATRIX.show()

---

## 4.11 Spending a 2.5-second budget

Bring a budget to the design review. It converts an argument about architecture into
arithmetic, and it usually settles the argument in one direction people find surprising.


In [ ]:
from raglab import costs

model = costs.latency_model(n_candidates=100, rerank_depth=50, out_tokens=450,
                            reranker="cross")
viz.budget(model["items"], total=model["target_ms"], unit="ms",
           title="Spending a 2.5 s p95 on one grounded answer",
           kicker="Engineering budget",
           caption="Generation dominates. Retrieval quality is nearly free in latency terms — "
                   "which is why cutting the reranker to save 200 ms is usually the wrong "
                   "trade.", source="Deck slide 48")

rows = []
for rr in ("none", "late", "cross", "llm"):
    m = costs.latency_model(reranker=rr, rerank_depth=50)
    rows.append([rr, round(m["subtotal_ms"]), f"{m['subtotal_ms']/model['target_ms']:.0%}",
                 "yes" if m["within_budget"] else "NO"])
tables.show(pd.DataFrame(rows, columns=["Reranker", "Total ms", "Share of 2.5 s budget",
                                        "Inside budget"]),
            title="What each reranker choice costs the budget",
            kicker="Latency", emphasize="Reranker",
            caption="Time-to-first-token is the number the user feels, so streaming changes "
                    "the perceived budget completely. Agentic loops multiply all of it — "
                    "three retrieval turns is three times the pre-generation cost plus three "
                    "generations.")

---

## 4.12 Failure points and the interview


In [ ]:
fp = []
ident_dense = pipeline.evaluate(pipe.variant("d", fusion="dense", rerank="none"), ident,
                                pipe.chunks, personas=bundle.personas)
ident_lex = pipeline.evaluate(pipe.variant("l", fusion="lexical", rerank="none"), ident,
                              pipe.chunks, personas=bundle.personas)
fp.append(["Identifier miss",
           f"dense {metrics.summarize(ident_dense)['evidence_recall']:.3f} vs lexical "
           f"{metrics.summarize(ident_lex)['evidence_recall']:.3f} on "
           f"{len(ident)} ERR_* questions",
           "Dense search retrieves related incidents but not the exact code. Keep the "
           "lexical leg, and keep the analyzer honest (§4.3)."])
fp.append(["Embedding mismatch",
           f"{metrics.summarize(correct)['evidence_recall']:.3f} → "
           f"{metrics.summarize(broken)['evidence_recall']:.3f} when the query prefix is "
           "dropped",
           "Prefixes belong in a config that both sides read, not in whichever script wrote "
           "the index."])
fp.append(["ANN loss",
           f"recall@20 {recall[0]:.2f} at efSearch=8 versus 1.00 at efSearch="
           f"{efs[recall.index(1.0)] if 1.0 in recall else efs[-1]}",
           "The approximate index does not return a gold chunk that exact search would find. "
           "Invisible in every downstream metric — measure against flat search on a sample."])
fp.append(["Reranker ceiling",
           f"0 of {len(ceilings)} questions where reranking exceeded Recall@N",
           "The gold document is not in the candidate pool, so reranking cannot recover it. "
           "Fix stage one first."])

tables.show(pd.DataFrame(fp, columns=["Signature", "Measured here", "What to do about it"]),
            title="Failure points: retrieval and reranking",
            kicker="Failure points", source="Deck slide 49", emphasize="Signature")

In [ ]:
tables.show(pd.DataFrame([
    [catalog.SECTION_QUESTIONS[4][0],
     "Whether you answer with a corpus property rather than a preference",
     "BM25 when identifiers, error codes, API names and exact terminology matter. Dense when "
     "query wording differs from corpus wording. Hybrid when both — and then say how you "
     "would weight it and how you would know the weight was wrong (§4.9)."],
    [catalog.SECTION_QUESTIONS[4][1],
     "Whether you can connect the algebra to an index setting",
     "cos = q·d/(‖q‖‖d‖); after L2 normalisation the denominator is 1, so cosine and inner "
     "product give identical rankings. Consequence: an index configured for inner product is "
     "only equivalent if you normalise on write."],
    [catalog.SECTION_QUESTIONS[4][2],
     "Whether you know what each buys and what each costs",
     "Cross-encoder: full attention over the pair, best quality, nothing precomputable, "
     "50–300 ms batched. Late interaction: MaxSim over precomputed token vectors, 10–40 ms, "
     "10–100× storage. The storage multiplier is the part people forget."],
    [catalog.SECTION_QUESTIONS[4][3],
     "Whether you check operational causes before model quality",
     "The seven-step checklist in §4.7, in order, with flat search as the ground truth that "
     "separates index loss from embedding loss. Starting at 'the model is worse' is the red "
     "flag."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: retrieval and reranking",
    kicker="Section 4 · interview", source="Deck slide 50", emphasize="Question")

---

## 4.13 Checkpoint

1. Your ANN index reports 0.99 recall in a benchmark and users still report missing results.
   Name two causes this notebook measured.
2. You have 200 labelled examples. Should you tune α?
3. A colleague proposes dropping the reranker to save 220 ms of a 2.5 s budget. Respond with
   numbers.


In [ ]:
print("1 ·  (a) The benchmark ran WITHOUT the production filters. A selective pre-filter")
print("         strands graph traversal — §4.6 measured the drop at the same efSearch.")
print("     (b) The benchmark measured recall against the ANN index's own top-N rather than")
print("         against flat search. Flat is the only ground truth.\n")

print("2 ·  Only with the frozen slice held back and a bootstrap interval on the delta.")
b = metrics.paired_bootstrap(legs["RRF (equal weight)"], legs[f"weighted α={best_a}"],
                             "evidence_recall")
print(f"     Here: RRF → weighted α={best_a} is {b['delta']:+.4f} evidence recall,")
print(f"     95% CI [{b['ci'][0]:+.4f}, {b['ci'][1]:+.4f}] → {b['verdict']}.")
print("     And the deck's caveat stands: an α tuned on 200 examples will not survive a")
print("     corpus refresh. Put a re-tune on a schedule or do not tune it at all.\n")

no_rr = metrics.summarize(comparison["no reranker · dev"])
with_rr = metrics.summarize(comparison["learned cross-encoder · dev"])
m_none = costs.latency_model(reranker="none")
m_cross = costs.latency_model(reranker="cross")
print("3 ·  The saving is real and small; the quality cost is real and larger.")
print(f"     latency        {m_cross['subtotal_ms']:.0f} ms → {m_none['subtotal_ms']:.0f} ms "
      f"({m_none['subtotal_ms']-m_cross['subtotal_ms']:+.0f} ms, "
      f"{(m_none['subtotal_ms']-m_cross['subtotal_ms'])/m_cross['target_ms']:+.1%} of budget)")
print(f"     evidence recall {with_rr['evidence_recall']:.3f} → "
      f"{no_rr['evidence_recall']:.3f} "
      f"({no_rr['evidence_recall']-with_rr['evidence_recall']:+.3f})")
print("     Generation dominates the budget. Buying back 9% of the latency envelope by")
print("     giving up 8 points of evidence recall is the worst trade on the deck's cost-lever")
print("     list, and it is worth saying so with the numbers rather than as an opinion.")

---

## What carries forward

- BM25 is three ideas, and one of them (`b`) is why re-chunking silently re-tunes your
  lexical index.
- The analyzer decides what is searchable. On a corpus with identifiers, the default is
  wrong and nothing will tell you.
- Dimension trades exactness against generalisation. Sweep it, slice the result, and do not
  inherit a default.
- ANN recall is a tunable, measured against flat search — *with your real filters on*.
- Fusion weights and rerankers are models. Fit them on dev, verify on frozen, and report a
  bootstrap interval rather than a headline.

**Next:** `05_llm_context_design.ipynb` — the token budget with hard caps, what a packed
context actually looks like, provenance that survives, and the position effect measured
rather than assumed.
